# 📊 Sistema de Assertividade Comercial — Meta Ads + Reuniões

Este notebook lê os relatórios de **tráfego pago da Meta (CSV)** e a sua **planilha de reuniões**, cruza as duas bases e gera uma planilha Excel completa com toda a assertividade do mês.

**Como usar (todo mês):**
1. Coloque o(s) CSV(s) exportados do Gerenciador de Anúncios em `dados/meta/`
2. Coloque a planilha de reuniões (`.xlsx` ou `.csv`) em `dados/reunioes/`
3. Ajuste a célula **⚙️ Configuração** se precisar (mês, metas, nomes de colunas)
4. Menu **Run → Run All Cells** (ou `Kernel → Restart & Run All`)
5. O relatório sai em `relatorios/Assertividade_Comercial_AAAA-MM.xlsx`

> O sistema reconhece automaticamente os nomes de colunas mais comuns (em português e inglês). Se alguma coluna da sua planilha tiver um nome diferente, basta acrescentar o nome na lista de apelidos da configuração.

## 1. Bibliotecas

In [ ]:
import re
import unicodedata
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from openpyxl import Workbook
from openpyxl.chart import BarChart, Reference
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 250)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
print("pandas", pd.__version__, "| numpy", np.__version__)

## 2. ⚙️ Configuração
Esta é a única célula que você normalmente precisa mexer.

In [ ]:
# ---------------- Pastas ----------------
PASTA_META      = Path("dados/meta")        # CSVs exportados do Gerenciador de Anúncios
PASTA_REUNIOES  = Path("dados/reunioes")    # planilha(s) de reuniões (.xlsx / .csv)
PASTA_SAIDA     = Path("relatorios")

# ---------------- Período ----------------
# "2026-09" para analisar setembro/2026. None = usa automaticamente o mês mais recente da planilha de reuniões.
MES_REFERENCIA = None

# ---------------- Metas (referências SUAS — ajuste ao seu negócio) ----------------
# Usadas só para marcar ✅ / ⚠️ no relatório e gerar o diagnóstico.
METAS = {
    "ctr":                0.010,   # CTR no link mínimo (1,0%)
    "cpl_max":            40.0,    # custo por lead máximo (R$)
    "lead_para_reuniao":  0.20,    # % dos leads que viram reunião agendada
    "comparecimento":     0.70,    # % de comparecimento nas reuniões
    "assertividade":      0.25,    # fechados / reuniões realizadas
    "win_rate":           0.40,    # fechados / (fechados + perdidos)
    "roas":               3.0,     # receita / investimento
    "cac_max":            800.0,   # custo de aquisição de cliente máximo (R$)
}

# ---------------- Planilha de reuniões ----------------
# Planilhas "de um mês só" (ex.: Controle Comercial - Setembro, com blocos Semana 1, Semana 2...):
# "auto" = se o arquivo tiver blocos de semana, NÃO filtra por data (muitas linhas não têm data preenchida)
#          e só aponta, na aba de qualidade, as datas que caem fora do mês.
FILTRAR_REUNIOES_POR_DATA = "auto"   # "auto", True ou False
# Ano usado quando a data vem sem ano (ex.: "31/08"). None = descobre pelas outras datas.
ANO_PADRAO = None
# Nomes com sufixo "Cliente - Fulano" (ex.: "Wagner - Parceiro"): o sufixo vira a coluna "Marcação".
# Liste aqui as marcações que NÃO vieram do tráfego pago (ex.: indicação de parceiro) — ficam fora do CAC.
MARCACOES_NAO_PAGAS = []             # ex.: ["Parceiro"]

# Leitura das observações: palavras que indicam sinais importantes (não mudam o status, só sinalizam)
SINAIS_OBSERVACAO = {
    "Provável perda":            ["nao tem interesse", "sem interesse", "trocou de contabilidade", "fechou com outr", "desistiu", "nao vai"],
    "Objeção: preço/orçamento":  ["caro", "pechinch", "orcamento", "cotando", "valores", "desconto", "quando receber", "recebesse um dinheiro", "fluxo de pagamento"],
    "Decide com cônjuge/sócio":  ["marido", "esposa", "mulher", "socio", "entre eles", "familia"],
    "Comparando concorrentes":   ["outras opcoes", "outra contabilidade", "concorren", "desenquadrad", "sua agencia"],
    "Promessa de fechamento":    ["vai fechar", "ira fechar", "fecha em", "fecharia", "disse que fechava", "quer fechar", "decidirao"],
    "Pediu retorno / follow-up": ["retornar", "retorne", "retorna", "falar em", "voltar a falar", "falar com ele", "falar no final", "follow", "semana", "outubro"],
    "Só tirou dúvidas":          ["tirar duvidas", "tirou duvidas"],
    "Pediu contrato/proposta":   ["contrato", "proposta"],
}
# Se True, reuniões "Reunião feita" cuja observação tem sinal de "Provável perda" passam a contar como Perdido.
CLASSIFICAR_PERDA_PELA_OBSERVACAO = False

# Probabilidade de fechamento usada para projetar o pipeline em negociação (forecast ponderado)
PROB_FECHAMENTO_NEGOCIACAO = 0.30
# Negociações abertas há mais dias que isso são marcadas como "parada"
DIAS_ALERTA_PIPELINE = 15

# ---------------- Origem dos leads ----------------
# Palavras que identificam que a reunião veio do tráfego pago (coluna "Origem" da sua planilha)
ORIGENS_TRAFEGO_PAGO = ["meta", "facebook", "instagram ads", "insta ads", "ads", "trafego", "anuncio", "fb", "pago"]
# Se a sua planilha NÃO tiver coluna de origem, considerar todas as reuniões como vindas do tráfego pago?
SEM_ORIGEM_CONSIDERAR_PAGO = True

# ---------------- Coluna de leads da Meta ----------------
# None = usa "Resultados". Se quiser forçar outra coluna (ex.: "Leads", "Conversas por mensagem iniciadas"), escreva o nome.
COLUNA_LEADS_META = None

# ---------------- Apelidos de colunas (o sistema procura na ordem) ----------------
COLUNAS_META = {
    "data_inicio":   ["Início dos relatórios", "Reporting starts", "Dia", "Day", "Data"],
    "data_fim":      ["Término dos relatórios", "Encerramento dos relatórios", "Reporting ends"],
    "campanha":      ["Nome da campanha", "Campaign name", "Campanha"],
    "conjunto":      ["Nome do conjunto de anúncios", "Ad set name", "Conjunto de anúncios"],
    "anuncio":       ["Nome do anúncio", "Ad name", "Anúncio"],
    "investimento":  ["Valor usado (BRL)", "Valor gasto (BRL)", "Amount spent (BRL)", "Valor usado", "Valor gasto", "Amount spent"],
    "impressoes":    ["Impressões", "Impressions"],
    "alcance":       ["Alcance", "Reach"],
    "cliques":       ["Cliques no link", "Link clicks", "Cliques (todos)", "Clicks (all)"],
    "resultados":    ["Resultados", "Results", "Leads", "Cadastros", "Conversas por mensagem iniciadas"],
    "tipo_resultado":["Indicador de resultado", "Tipo de resultado", "Result indicator", "Result type"],
}
COLUNAS_REUNIOES = {
    "data_reuniao":    ["Data da reunião", "Data da Reunião Realizada", "Data reuniao", "Data", "Dia"],
    "data_agendamento":["Data do Agendamento", "Data agendamento", "Data agendada", "Agendado em"],
    "cliente":         ["Cliente", "Nome", "Lead", "Empresa", "Nome do cliente"],
    "status":          ["Status", "Etapa do Funil", "Situação", "Situacao", "Resultado", "Etapa", "Andamento"],
    "valor":           ["Valor", "Valor do contrato", "Valor fechado", "Ticket", "Valor da proposta", "Valor (R$)"],
    "responsavel":     ["Responsável", "Responsavel", "Quem fez", "Vendedor", "Closer", "Consultor", "SDR"],
    "origem":          ["Origem", "Canal", "Fonte", "Origem do lead"],
    "campanha":        ["Campanha", "Nome da campanha", "Anúncio de origem"],
    "produto":         ["Produto", "Serviço", "Servico", "Plano", "Oferta"],
    "data_fechamento": ["Data de fechamento", "Data fechamento", "Data do fechamento", "Fechado em"],
    "data_lead":       ["Data do lead", "Data de entrada", "Data do primeiro contato", "Entrada"],
    "motivo_perda":    ["Motivo da perda", "Motivo", "Objeção", "Objecao", "Por que não fechou"],
    "observacoes":     ["Observações", "Observacoes", "Obs", "Comentários", "Anotações"],
}

# ---------------- Classificação dos status ----------------
# O texto livre da sua planilha é convertido em um status padrão. A ORDEM importa
# ("não fechou" precisa ser testado antes de "fechou").
REGRAS_STATUS = [
    ("No-show",       ["no show", "noshow", "no-show", "nao compareceu", "nao apareceu", "faltou", "ausente", "furou"]),
    ("Remarcado",     ["remarc", "reagend", "adiad"]),
    ("Cancelado",     ["cancel"]),
    ("Perdido",       ["nao fechou", "perdid", "perdeu", "recusou", "desistiu", "sem interesse", "nao tem interesse",
                       "descartad", "sem fit", "nao qualificad", "desqualificad", "reprovad", "nao vai fechar"]),
    ("Fechado",       ["fechou", "fechad", "ganho", "ganhou", "vendid", "venda realizada", "contrato assinado",
                       "assinou", "convertid", "pago", "cliente ativo"]),
    ("Em negociação", ["em processo", "processo", "negoci", "proposta", "follow", "aguardando", "pensando",
                       "andamento", "retorno", "analise", "avaliando", "quente", "morno"]),
    ("Reunião feita", ["reuniao feita", "reuniao realizada", "realizada", "call feita", "atendid"]),
    ("Agendado",      ["agendad", "marcad", "confirmad", "a realizar", "futura"]),
]
# Correções manuais exatas (texto da planilha -> status padrão). Ex.: {"talvez": "Em negociação"}
STATUS_MANUAL = {}

## 3. Funções de apoio (leitura, limpeza e padronização)

In [ ]:
def normalizar(txt):
    # minúsculo, sem acentos e sem espaços duplicados — base de todas as comparações de texto
    if txt is None or (isinstance(txt, float) and np.isnan(txt)):
        return ""
    txt = unicodedata.normalize("NFKD", str(txt).strip().lower()).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", txt)


import hashlib

def ler_bruto(caminho):
    # Lê o arquivo SEM assumir que a 1ª linha é o cabeçalho (tudo como texto)
    caminho = Path(caminho)
    if caminho.suffix.lower() in (".xlsx", ".xlsm", ".xls"):
        return pd.read_excel(caminho, header=None, dtype=object)
    for enc in ("utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(caminho, sep=None, engine="python", encoding=enc, header=None, dtype=str,
                               skip_blank_lines=False, keep_default_na=False, na_values=[""])
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Não consegui ler {caminho}")


def eh_vazia(linha):
    return all(pd.isna(v) or str(v).strip() == "" for v in linha)


def montar_tabela(bruto, apelidos_obrigatorios):
    # Encontra o(s) cabeçalho(s) dentro do arquivo e monta uma tabela única.
    #  - Tabela simples: cabeçalho na 1ª linha.
    #  - Painel em blocos (Semana 1, Semana 2...): cada bloco tem seu cabeçalho; lê até a próxima linha vazia
    #    e guarda de qual semana veio a linha. Totais/legendas abaixo dos blocos são ignorados.
    alvos = [[normalizar(a) for a in grupo] for grupo in apelidos_obrigatorios]
    def eh_cabecalho(linha):
        cels = [normalizar(v) for v in linha if pd.notna(v)]
        return all(any(c == a or c.startswith(a) for c in cels for a in grupo) for grupo in alvos)
    linhas = bruto.values.tolist()
    cabecalhos = [i for i, l in enumerate(linhas) if eh_cabecalho(l)]
    if not cabecalhos:
        return None, False
    partes = []
    for i in cabecalhos:
        nomes = [str(v).strip() if pd.notna(v) and str(v).strip() else f"coluna_{j+1}" for j, v in enumerate(linhas[i])]
        semana = None
        for k in range(i - 1, max(i - 4, -1), -1):          # título do bloco logo acima do cabeçalho
            t = str(linhas[k][0]).strip() if pd.notna(linhas[k][0]) else ""
            if re.match(r"(?i)^semana\s*\d+", t):
                semana = t; break
        dados = []
        for l in linhas[i + 1:]:
            if eh_vazia(l) or eh_cabecalho(l):
                break
            dados.append(l)
        bloco = pd.DataFrame(dados, columns=nomes)
        bloco["semana_planilha"] = semana
        partes.append(bloco)
    em_blocos = len(cabecalhos) > 1 or cabecalhos[0] > 0
    # blocos podem ter colunas extras (ex.: "Quem fez" só na Semana 1) -> une pelo nome
    return pd.concat(partes, ignore_index=True), em_blocos


def ler_pasta(pasta, extensoes, apelidos_obrigatorios):
    arquivos = sorted(p for p in Path(pasta).glob("*") if p.suffix.lower() in extensoes and not p.name.startswith("~$"))
    if not arquivos:
        raise FileNotFoundError(f"Nenhum arquivo {extensoes} encontrado em '{pasta}'.")
    partes, vistos, em_blocos = [], {}, False
    for arq in arquivos:
        h = hashlib.md5(arq.read_bytes()).hexdigest()
        if h in vistos:
            print(f"  ⚠️ {arq.name} é IDÊNTICO a {vistos[h]} — ignorado para não duplicar números")
            AVISOS.append(f"Arquivo '{arq.name}' é idêntico a '{vistos[h]}' e foi ignorado (evita contar investimento/reuniões em dobro).")
            continue
        vistos[h] = arq.name
        df, blocos = montar_tabela(ler_bruto(arq), apelidos_obrigatorios)
        if df is None:
            print(f"  ✖ {arq.name}: não achei o cabeçalho — ignorado")
            AVISOS.append(f"Arquivo '{arq.name}' ignorado: nenhum cabeçalho reconhecido.")
            continue
        em_blocos |= blocos
        df["arquivo_origem"] = arq.name
        partes.append(df)
        print(f"  ✔ {arq.name}: {len(df)} linhas de dados" + (" (painel em blocos por semana)" if blocos else ""))
    if not partes:
        raise ValueError(f"Nenhum arquivo válido em '{pasta}'.")
    return pd.concat(partes, ignore_index=True), em_blocos

AVISOS = []   # avisos de leitura que vão para o diagnóstico


def encontrar_coluna(df, apelidos):
    cols = {normalizar(c): c for c in df.columns}
    for a in apelidos:                       # 1º: nome exato
        if normalizar(a) in cols:
            return cols[normalizar(a)]
    for a in apelidos:                       # 2º: começa com o apelido
        for nc, c in cols.items():
            if nc.startswith(normalizar(a)):
                return c
    return None


def padronizar(df, mapa, nome_base):
    novo, log = pd.DataFrame(index=df.index), []
    usadas = set()
    for padrao, apelidos in mapa.items():
        col = encontrar_coluna(df.drop(columns=list(usadas), errors="ignore"), apelidos)
        if col is not None:
            novo[padrao] = df[col]
            usadas.add(col)
        log.append({"Base": nome_base, "Campo do sistema": padrao,
                    "Coluna encontrada no arquivo": col if col else "— NÃO ENCONTRADA —"})
    extras = [c for c in df.columns if c not in usadas and c not in ("arquivo_origem", "semana_planilha")]
    for c in extras:
        log.append({"Base": nome_base, "Campo do sistema": "(não usada)", "Coluna encontrada no arquivo": c})
    extras = [c for c in extras if c != "semana_planilha"]
    for c in ["arquivo_origem", "semana_planilha"]:
        if c in df:
            novo[c] = df[c]
    return novo, pd.DataFrame(log)


def para_numero(serie):
    # Converte 'R$ 1.500,00', '1,5', '12%', 1500 etc. em float
    if pd.api.types.is_numeric_dtype(serie):
        return serie.astype(float)
    def conv(v):
        if pd.isna(v):
            return np.nan
        if isinstance(v, (int, float, np.number)):
            return float(v)
        s = re.sub(r"[^\d,.\-]", "", str(v))
        if s in ("", "-", ".", ","):
            return np.nan
        if "," in s and "." in s:
            s = s.replace(".", "").replace(",", ".") if s.rfind(",") > s.rfind(".") else s.replace(",", "")
        elif s.count(",") > 1:
            s = s.replace(",", "")
        elif "," in s:
            s = s.replace(",", ".")
        elif re.fullmatch(r"-?[1-9]\d{0,2}(\.\d{3})+", s):
            s = s.replace(".", "")
        try:
            return float(s)
        except ValueError:
            return np.nan
    return serie.map(conv).astype(float)


def para_data(serie, ano_padrao=None):
    if pd.api.types.is_datetime64_any_dtype(serie):
        return serie.dt.normalize()
    txt = serie.astype("string").str.replace(r"\s+", "", regex=True).str.rstrip("/")   # "08/09/ 2026" e "17/09/"
    if ano_padrao:
        sem_ano = txt.str.fullmatch(r"\d{1,2}/\d{1,2}").fillna(False)                 # "31/08"
        txt = txt.where(~sem_ano, txt + f"/{ano_padrao}")
    iso = txt.str.match(r"^\d{4}-\d{2}-\d{2}").fillna(False)
    out = pd.Series(pd.NaT, index=serie.index, dtype="datetime64[ns]")
    if iso.any():
        out[iso] = pd.to_datetime(txt[iso].str[:10], format="%Y-%m-%d", errors="coerce")
    if (~iso).any():
        out[~iso] = pd.to_datetime(txt[~iso], dayfirst=True, format="mixed", errors="coerce")
    return out.dt.normalize()


def classificar_status(txt):
    if pd.isna(txt) or str(txt).strip() == "":
        return "Sem status"
    if txt in STATUS_MANUAL:
        return STATUS_MANUAL[txt]
    n = normalizar(txt)
    for categoria, chaves in REGRAS_STATUS:
        if any(normalizar(k) in n for k in chaves):
            return categoria
    return "Não classificado"


def div(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    with np.errstate(divide="ignore", invalid="ignore"):
        r = np.where(b != 0, a / b, 0.0)
    return r.item() if r.ndim == 0 else r

print("Funções carregadas ✔")

## 4. Leitura da base da Meta (tráfego pago)

In [ ]:
print("Lendo Meta Ads:")
meta_bruto, _ = ler_pasta(PASTA_META, {".csv", ".xlsx"}, [COLUNAS_META["campanha"], COLUNAS_META["investimento"]])
if COLUNA_LEADS_META:
    COLUNAS_META["resultados"] = [COLUNA_LEADS_META]
meta, log_meta = padronizar(meta_bruto, COLUNAS_META, "Meta Ads")

for c in ["investimento", "impressoes", "alcance", "cliques", "resultados"]:
    meta[c] = para_numero(meta[c]) if c in meta else 0.0
    meta[c] = meta[c].fillna(0.0)
for c in ["data_inicio", "data_fim"]:
    if c in meta:
        meta[c] = para_data(meta[c])
for c in ["campanha", "conjunto", "anuncio", "tipo_resultado"]:
    if c not in meta:
        meta[c] = "(sem informação)"
    meta[c] = meta[c].fillna("(sem informação)").astype(str)

# remove linhas vazias / linha de totais que algumas exportações trazem
meta = meta[~((meta["campanha"].str.strip().isin(["", "(sem informação)", "nan"])) & (meta["investimento"] == 0))]
meta["tipo_resultado"] = meta["tipo_resultado"].str.replace("actions:", "", regex=False).replace(
    {"onsite_conversion.messaging_conversation_started_7d": "Conversas no WhatsApp/Direct iniciadas", "lead": "Leads (formulário)",
     "nan": "(sem resultado)"})
# campanhas pausadas que não gastaram nem apareceram no período só poluem as tabelas
sem_entrega = meta[(meta["investimento"] == 0) & (meta["impressoes"] == 0)]
meta = meta.drop(sem_entrega.index)
if len(sem_entrega):
    print(f"  ℹ️ {len(sem_entrega)} campanhas sem gasto e sem impressões no período foram deixadas de fora.")
print(f"\nBase Meta tratada: {len(meta)} linhas | Investimento total R$ {meta['investimento'].sum():,.2f}")
display(log_meta[log_meta["Campo do sistema"] != "(não usada)"])

## 5. Leitura da planilha de reuniões

In [ ]:
print("Lendo reuniões:")
reun_bruto, REUNIOES_EM_BLOCOS = ler_pasta(PASTA_REUNIOES, {".xlsx", ".xlsm", ".xls", ".csv"},
                                          [COLUNAS_REUNIOES["cliente"], COLUNAS_REUNIOES["status"]])
reun, log_reun = padronizar(reun_bruto, COLUNAS_REUNIOES, "Reuniões")
reun = reun[reun["cliente"].notna() & (reun["cliente"].astype(str).str.strip() != "")].reset_index(drop=True)

if "status" not in reun:
    raise KeyError("Não encontrei a coluna de STATUS na planilha de reuniões. Adicione o nome dela em COLUNAS_REUNIOES['status'].")
if "data_reuniao" not in reun and "data_agendamento" not in reun:
    raise KeyError("Não encontrei a coluna de DATA da reunião. Adicione o nome dela em COLUNAS_REUNIOES['data_reuniao'].")

if ANO_PADRAO is None:   # descobre o ano pelas datas completas
    anos = pd.concat([reun[c].astype("string").str.extract(r"(20\d{2})")[0] for c in
                      ["data_reuniao", "data_agendamento", "data_fechamento", "data_lead"] if c in reun]).dropna()
    ANO_PADRAO = int(anos.mode()[0]) if len(anos) else datetime.now().year
reun["data_reuniao_texto"] = reun["data_reuniao"].astype("string") if "data_reuniao" in reun else pd.NA
for c in ["data_reuniao", "data_agendamento", "data_fechamento", "data_lead"]:
    if c in reun:
        reun[c] = para_data(reun[c], ANO_PADRAO)
if "data_agendamento" not in reun:
    reun["data_agendamento"] = pd.NaT
# data usada no filtro/semana: data da reunião; se vazia, a do agendamento
reun["data_referencia"] = reun["data_reuniao"].fillna(reun["data_agendamento"])
TEM_VALOR = "valor" in reun and para_numero(reun["valor"]).notna().any()
reun["valor"] = para_numero(reun["valor"]) if "valor" in reun else np.nan
for c in ["cliente", "responsavel", "origem", "campanha", "produto", "motivo_perda", "observacoes"]:
    if c not in reun:
        reun[c] = np.nan
    reun[c] = reun[c].astype("string").str.strip().replace({"": pd.NA, "nan": pd.NA})

reun["status_original"] = reun["status"].astype("string").str.strip()
reun["status_padrao"] = reun["status_original"].map(classificar_status)

# "Cliente - Fulano" -> Marcação = Fulano (parceiro, indicação, vendedor... você define o significado)
partes_nome = reun["cliente"].str.extract(r"^\s*(.*?)\s*-\s*([^-]+?)\s*$")
reun["marcacao"] = partes_nome[1].fillna("(sem marcação)")
reun["cliente"] = partes_nome[0].fillna(reun["cliente"]).str.strip()

# Leitura das observações (sinais)
def sinais(obs):
    n = normalizar(obs)
    return [rotulo for rotulo, chaves in SINAIS_OBSERVACAO.items() if n and any(normalizar(k) in n for k in chaves)]
reun["sinais_obs"] = reun["observacoes"].map(sinais)
reun["sinais_texto"] = reun["sinais_obs"].map(", ".join).replace("", pd.NA)
if CLASSIFICAR_PERDA_PELA_OBSERVACAO:
    mudar = (reun["status_padrao"] == "Reunião feita") & reun["sinais_obs"].map(lambda l: "Provável perda" in l)
    reun.loc[mudar, "status_padrao"] = "Perdido"
    reun.loc[mudar & reun["motivo_perda"].isna(), "motivo_perda"] = reun.loc[mudar, "observacoes"]
    print(f"  {mudar.sum()} reuniões reclassificadas como Perdido pela observação")

# origem paga x não paga
tem_origem = reun["origem"].notna().any()
if tem_origem:
    padroes = [normalizar(p) for p in ORIGENS_TRAFEGO_PAGO]
    reun["trafego_pago"] = reun["origem"].map(lambda o: any(p in normalizar(o) for p in padroes) if pd.notna(o) else False)
    reun["trafego_pago"] |= reun["campanha"].notna()
else:
    reun["trafego_pago"] = SEM_ORIGEM_CONSIDERAR_PAGO
nao_pagas = [normalizar(x) for x in MARCACOES_NAO_PAGAS]
reun.loc[reun["marcacao"].map(normalizar).isin(nao_pagas), "trafego_pago"] = False
reun["origem"] = reun["origem"].fillna("(sem origem)")
reun["responsavel"] = reun["responsavel"].fillna("(sem responsável)")
reun["produto"] = reun["produto"].fillna("(sem produto)")

print(f"\nBase de reuniões tratada: {len(reun)} linhas")
display(log_reun[log_reun["Campo do sistema"] != "(não usada)"])
print("\nComo cada status escrito na planilha foi interpretado:")
display(reun.groupby(["status_original", "status_padrao"], dropna=False).size().rename("Qtd").reset_index()
            .sort_values(["status_padrao", "Qtd"], ascending=[True, False]))

## 6. Filtro do mês de referência

In [ ]:
if MES_REFERENCIA is None:
    meses = reun["data_referencia"].dropna().dt.to_period("M")
    MES_REFERENCIA = (meses.mode()[0] if REUNIOES_EM_BLOCOS else meses.max()).strftime("%Y-%m")
periodo = pd.Period(MES_REFERENCIA, "M")
ini_mes, fim_mes = periodo.start_time.normalize(), periodo.end_time.normalize()
MESES_PT = ["janeiro", "fevereiro", "março", "abril", "maio", "junho", "julho", "agosto",
            "setembro", "outubro", "novembro", "dezembro"]
NOME_MES = f"{MESES_PT[periodo.month - 1]}/{periodo.year}"

filtrar = (not REUNIOES_EM_BLOCOS) if FILTRAR_REUNIOES_POR_DATA == "auto" else bool(FILTRAR_REUNIOES_POR_DATA)
dref = reun["data_referencia"]
sem_data = reun[dref.isna()]
fora_mes = reun[dref.notna() & (dref.dt.to_period("M") != periodo)]
if filtrar:
    r = reun[dref.dt.to_period("M") == periodo].copy()
    print("Filtro por data: ligado (linhas fora do mês e sem data ficam de fora)")
else:
    r = reun.copy()
    print("Filtro por data: desligado — a planilha já é do mês (painel por semanas). Todas as linhas entram;")
    print("datas fora do mês são só apontadas na aba 'Qualidade dos Dados'.")

# Meta: se a exportação é diária, filtra; se é um relatório de período, confere a cobertura
aviso_meta = ""
m = meta.copy()
if "data_inicio" in m and m["data_inicio"].notna().any():
    diario = "data_fim" not in m or (m["data_inicio"] == m.get("data_fim", m["data_inicio"])).all()
    if diario:
        m = m[m["data_inicio"].dt.to_period("M") == periodo]
    else:
        ini_rel, fim_rel = m["data_inicio"].min(), m["data_fim"].max()
        if ini_rel > ini_mes or fim_rel < fim_mes or ini_rel.to_period("M") != periodo:
            aviso_meta = (f"O relatório da Meta cobre {ini_rel:%d/%m/%Y} a {fim_rel:%d/%m/%Y}, "
                          f"diferente do mês analisado ({ini_mes:%d/%m/%Y} a {fim_mes:%d/%m/%Y}).")
print(f"Mês analisado: {NOME_MES}")
print(f"Reuniões analisadas: {len(r)} | com data fora do mês: {len(fora_mes)} | sem data: {len(sem_data)}")
print(f"Linhas Meta no período: {len(m)}")
if aviso_meta:
    print("⚠️", aviso_meta)

## 7. Motor de métricas
Cada taxa é definida uma única vez (numerador ÷ denominador). A mesma definição é usada para calcular no notebook **e** para escrever a fórmula viva no Excel.

In [ ]:
STATUS_REALIZADA = ["Fechado", "Perdido", "Em negociação", "Reunião feita"]
STATUS_ORDEM = ["Fechado", "Em negociação", "Reunião feita", "Perdido", "No-show", "Remarcado", "Cancelado",
                "Agendado", "Não classificado", "Sem status"]

def g(serie, chave):   # pega um valor da série, 0 se a coluna não existir (ex.: planilha sem valor)
    return serie.get(chave, 0.0)

def contagens(df):
    # Contagens base de um grupo de reuniões
    s = df["status_padrao"]
    fechados = df[s == "Fechado"]
    neg = df[s == "Em negociação"]
    base = {
        "Reuniões agendadas":       len(df),
        "Reuniões realizadas":      s.isin(STATUS_REALIZADA).sum(),
        "Fechados":                 (s == "Fechado").sum(),
        "Em negociação":            (s == "Em negociação").sum(),
        "Reunião feita sem desfecho": (s == "Reunião feita").sum(),
        "Perdidos":                 (s == "Perdido").sum(),
        "No-show":                  (s == "No-show").sum(),
        "Remarcadas":               (s == "Remarcado").sum(),
        "Canceladas":               (s == "Cancelado").sum(),
        "Ainda vão acontecer":      (s == "Agendado").sum(),
        "Não classificadas":        s.isin(["Não classificado", "Sem status"]).sum(),
        "Receita fechada (R$)":     fechados["valor"].sum(),
        "Valor em negociação (R$)": neg["valor"].sum(),
        "Valor perdido (R$)":       df.loc[s == "Perdido", "valor"].sum(),
    }
    if not TEM_VALOR:   # planilha sem coluna de valor: não inventa receita zerada
        base = {k: v for k, v in base.items() if "(R$)" not in k}
    return pd.Series(base)

TAXAS_COMERCIAIS = [
    {"nome": "Comparecimento (%)",               "num": ["Reuniões realizadas"], "den": ["Reuniões realizadas", "No-show"]},
    {"nome": "Agendamento → realização (%)",     "num": ["Reuniões realizadas"], "den": ["Reuniões realizadas", "No-show", "Remarcadas", "Canceladas"]},
    {"nome": "Assertividade - fechamento s/ realizadas (%)", "num": ["Fechados"], "den": ["Reuniões realizadas"]},
    {"nome": "Win rate - fechados s/ decididas (%)", "num": ["Fechados"], "den": ["Fechados", "Perdidos"]},
    {"nome": "Fechamento s/ agendadas (%)",      "num": ["Fechados"], "den": ["Reuniões agendadas"]},
    {"nome": "Taxa de perda (%)",                "num": ["Perdidos"], "den": ["Reuniões realizadas"]},
    {"nome": "Ainda em negociação (%)",          "num": ["Em negociação"], "den": ["Reuniões realizadas"]},
    {"nome": "Sem desfecho registrado (%)",      "num": ["Reunião feita sem desfecho"], "den": ["Reuniões realizadas"]},
    {"nome": "No-show (%)",                      "num": ["No-show"], "den": ["Reuniões agendadas"]},
    {"nome": "Ticket médio (R$)",                "num": ["Receita fechada (R$)"], "den": ["Fechados"]},
    {"nome": "Participação na receita (%)",      "num": ["Receita fechada (R$)"], "den": "__TOTAL__"},
]

TAXAS_META = [
    {"nome": "% do investimento (%)",            "num": ["Investimento (R$)"], "den": "__TOTAL__"},
    {"nome": "CTR no link (%)",                  "num": ["Cliques no link"], "den": ["Impressões"]},
    {"nome": "CPC (R$)",                         "num": ["Investimento (R$)"], "den": ["Cliques no link"]},
    {"nome": "CPM (R$)",                         "num": ["Investimento (R$)"], "den": ["Impressões"], "mult": 1000},
    {"nome": "Frequência (x)",                   "num": ["Impressões"], "den": ["Alcance"]},
    {"nome": "Clique → lead (%)",                "num": ["Leads / resultados"], "den": ["Cliques no link"]},
    {"nome": "Custo por lead (R$)",              "num": ["Investimento (R$)"], "den": ["Leads / resultados"]},
]

TAXAS_CRUZADAS = [
    {"nome": "Custo por lead (R$)",              "num": ["Investimento (R$)"], "den": ["Leads Meta"]},
    {"nome": "Lead → reunião (%)",               "num": ["Reuniões agendadas"], "den": ["Leads Meta"]},
    {"nome": "Comparecimento (%)",               "num": ["Reuniões realizadas"], "den": ["Reuniões realizadas", "No-show"]},
    {"nome": "Assertividade (%)",                "num": ["Fechados"], "den": ["Reuniões realizadas"]},
    {"nome": "Lead → cliente (%)",               "num": ["Fechados"], "den": ["Leads Meta"]},
    {"nome": "Custo por reunião realizada (R$)", "num": ["Investimento (R$)"], "den": ["Reuniões realizadas"]},
    {"nome": "CAC (R$)",                         "num": ["Investimento (R$)"], "den": ["Fechados"]},
    {"nome": "ROAS (x)",                         "num": ["Receita fechada (R$)"], "den": ["Investimento (R$)"]},
]


def aplicar_taxas(df, taxas, linha_total=None):
    df = df.copy()
    for t in taxas:
        if not all(c in df for c in t["num"]):
            continue
        num = df[t["num"]].sum(axis=1)
        if t["den"] == "__TOTAL__":
            den = num.sum() if linha_total is None else linha_total
        else:
            if not all(c in df for c in t["den"]):
                continue
            den = df[t["den"]].sum(axis=1)
        df[t["nome"]] = div(num, den) * t.get("mult", 1)
    return df


def tabela_comercial(df, coluna, rotulo, ordem=None):
    if df.empty:
        return pd.DataFrame(columns=[rotulo])
    t = df.groupby(coluna, dropna=False).apply(contagens, include_groups=False)
    t = t.reset_index().rename(columns={coluna: rotulo})
    if ordem is not None:
        t[rotulo] = pd.Categorical(t[rotulo], ordem, ordered=True)
        t = t.sort_values(rotulo)
        t[rotulo] = t[rotulo].astype(str)
    else:
        t = t.sort_values(["Fechados", "Reuniões agendadas"], ascending=False)
    return t.reset_index(drop=True)

print("Motor de métricas pronto ✔")

## 8. Métricas da Meta (tráfego pago)

In [ ]:
def agrupar_meta(df, chaves):
    g = df.groupby(chaves, dropna=False).agg(**{
        "Investimento (R$)": ("investimento", "sum"),
        "Impressões": ("impressoes", "sum"),
        "Alcance": ("alcance", "sum"),
        "Cliques no link": ("cliques", "sum"),
        "Leads / resultados": ("resultados", "sum"),
        "Tipo de resultado": ("tipo_resultado", lambda s: ", ".join(sorted(set(s)))),
    }).reset_index()
    return g.sort_values("Investimento (R$)", ascending=False).reset_index(drop=True)

meta_camp = agrupar_meta(m, ["campanha"]).rename(columns={"campanha": "Campanha"})
meta_conj = agrupar_meta(m, ["campanha", "conjunto"]).rename(columns={"campanha": "Campanha", "conjunto": "Conjunto de anúncios"})
meta_anun = agrupar_meta(m, ["campanha", "conjunto", "anuncio"]).rename(
    columns={"campanha": "Campanha", "conjunto": "Conjunto de anúncios", "anuncio": "Anúncio"})

INV = m["investimento"].sum()
IMP = m["impressoes"].sum()
ALC = m["alcance"].sum()
CLQ = m["cliques"].sum()
LEADS = m["resultados"].sum()
tipos_resultado = sorted(m["tipo_resultado"].unique())

print(f"Investimento: R$ {INV:,.2f} | Impressões: {IMP:,.0f} | Cliques: {CLQ:,.0f} | Leads/resultados: {LEADS:,.0f}")
print(f"CTR {div(CLQ, IMP):.2%} | CPC R$ {div(INV, CLQ):,.2f} | CPM R$ {div(INV, IMP) * 1000:,.2f} | CPL R$ {div(INV, LEADS):,.2f}")
if len(tipos_resultado) > 1:
    print("⚠️ Atenção: há tipos de resultado diferentes somados em 'Leads':", tipos_resultado)
display(aplicar_taxas(meta_camp, TAXAS_META, INV))

## 9. Métricas comerciais (reuniões)

In [ ]:
geral = contagens(r)
rp = r[r["trafego_pago"]]
pago = contagens(rp)

if "semana_planilha" in r and r["semana_planilha"].notna().any():
    r["semana"] = r["semana_planilha"].fillna("(sem semana)")          # usa os blocos da própria planilha
else:
    r["semana"] = r["data_referencia"].dt.to_period("W-SUN").map(
        lambda p: f"{p.start_time:%d/%m} a {p.end_time:%d/%m}" if pd.notna(p) else "(sem data)")
DIAS_PT = ["1-Segunda", "2-Terça", "3-Quarta", "4-Quinta", "5-Sexta", "6-Sábado", "7-Domingo"]
r["dia_semana"] = r["data_reuniao"].dt.dayofweek.map(lambda d: DIAS_PT[int(d)] if pd.notna(d) else "(sem data)")
r["status_ord"] = r["status_padrao"]

por_status = (r.groupby("status_padrao").agg(**{"Quantidade": ("status_padrao", "size"), "Valor total (R$)": ("valor", "sum")})
               .reindex([s for s in STATUS_ORDEM if s in r["status_padrao"].unique()]).reset_index()
               .rename(columns={"status_padrao": "Status padronizado"}))
por_status["% do total (%)"] = div(por_status["Quantidade"], por_status["Quantidade"].sum())
if not TEM_VALOR:
    por_status = por_status.drop(columns="Valor total (R$)")

por_resp    = tabela_comercial(r, "responsavel", "Responsável")
por_origem  = tabela_comercial(r, "origem", "Origem")
por_produto = tabela_comercial(r, "produto", "Produto") if (r["produto"] != "(sem produto)").any() else pd.DataFrame()
por_marcacao = tabela_comercial(r, "marcacao", "Marcação no nome") if (r["marcacao"] != "(sem marcação)").any() else pd.DataFrame()
ordem_sem = list(dict.fromkeys(r["semana"])) if REUNIOES_EM_BLOCOS else list(dict.fromkeys(r.sort_values("data_referencia")["semana"]))
por_semana  = tabela_comercial(r, "semana", "Semana", ordem=ordem_sem)
por_dia     = tabela_comercial(r, "dia_semana", "Dia da semana", ordem=DIAS_PT + ["(sem data)"])

# Status original x padronizado (transparência total da classificação)
mapa_status = (r.groupby(["status_padrao", "status_original"], dropna=False).size().rename("Quantidade").reset_index()
                .rename(columns={"status_padrao": "Status padronizado", "status_original": "Texto escrito na planilha"}))

# Status x responsável (matriz)
matriz_resp = pd.crosstab(r["responsavel"], r["status_padrao"]).reindex(columns=[s for s in STATUS_ORDEM if s in r["status_padrao"].unique()], fill_value=0)
matriz_resp = matriz_resp.reset_index().rename(columns={"responsavel": "Responsável"})

print(f"Agendadas {geral['Reuniões agendadas']:.0f} | Realizadas {geral['Reuniões realizadas']:.0f} | "
      f"Fechados {geral['Fechados']:.0f} | Perdidos {geral['Perdidos']:.0f} | Em negociação {geral['Em negociação']:.0f}")
print(f"Assertividade (fechados ÷ realizadas): {div(geral['Fechados'], geral['Reuniões realizadas']):.1%}")
print(f"Receita fechada: R$ {g(geral, 'Receita fechada (R$)'):,.2f}" if TEM_VALOR else "Receita: planilha sem coluna de valor")
display(por_status)
display(aplicar_taxas(por_resp, TAXAS_COMERCIAIS, g(geral, "Receita fechada (R$)")))

## 10. Ciclo de venda, motivos de perda e pipeline em aberto

In [ ]:
# ---- Ciclo de venda: dias entre o 1º contato (ou reunião) e o fechamento
fech = r[r["status_padrao"] == "Fechado"].copy()
ciclo = pd.DataFrame()
ciclo_resp = pd.DataFrame()
if "data_fechamento" in fech and fech["data_fechamento"].notna().any():
    base_ini = fech["data_lead"] if "data_lead" in fech and fech["data_lead"].notna().any() else fech["data_reuniao"]
    fech["Dias até fechar"] = (fech["data_fechamento"] - base_ini).dt.days
    d = fech["Dias até fechar"].dropna()
    ciclo = pd.DataFrame({"Indicador": ["Fechamentos com data", "Média (dias)", "Mediana (dias)", "Mais rápido (dias)",
                                        "Mais lento (dias)", "Fechou no mesmo dia da reunião", "Fechou em até 7 dias", "Fechou em mais de 7 dias"],
                          "Valor": [len(d), d.mean(), d.median(), d.min(), d.max(), (d == 0).sum(), ((d > 0) & (d <= 7)).sum(), (d > 7).sum()]})
    ciclo_resp = (fech.groupby("responsavel")["Dias até fechar"].agg(["count", "mean", "median", "min", "max"]).reset_index()
                  .rename(columns={"responsavel": "Responsável", "count": "Fechamentos", "mean": "Média (dias)",
                                   "median": "Mediana (dias)", "min": "Mínimo (dias)", "max": "Máximo (dias)"}))
    print("Ciclo de venda calculado ✔")
else:
    print("Sem coluna de data de fechamento — ciclo de venda não calculado.")

# ---- Tempo entre o agendamento e a reunião acontecer
ag = r[r["data_agendamento"].notna() & r["data_reuniao"].notna()].copy()
ag["Dias agendamento → reunião"] = (ag["data_reuniao"] - ag["data_agendamento"]).dt.days
tempo_agenda = pd.DataFrame()
if len(ag):
    d = ag["Dias agendamento → reunião"]
    tempo_agenda = pd.DataFrame({"Indicador": ["Reuniões com as duas datas", "Média (dias)", "Mediana (dias)", "Reunião no mesmo dia do agendamento",
                                               "1 a 2 dias depois", "3 dias ou mais"],
                                 "Valor": [len(d), d.mean(), d.median(), (d == 0).sum(), d.between(1, 2).sum(), (d >= 3).sum()]})
    print(f"Agendamento → reunião: média {d.mean():.1f} dias ({len(d)} reuniões com as duas datas)")

# ---- Motivos de perda
perd = r[r["status_padrao"] == "Perdido"].copy()
perd["motivo_perda"] = perd["motivo_perda"].fillna("(motivo não informado)")
motivos = (perd.groupby("motivo_perda").agg(**{"Quantidade": ("motivo_perda", "size"), "Valor potencial perdido (R$)": ("valor", "sum")})
               .sort_values("Quantidade", ascending=False).reset_index().rename(columns={"motivo_perda": "Motivo"}))
motivos["% das perdas (%)"] = div(motivos["Quantidade"], motivos["Quantidade"].sum())

# ---- Pipeline em negociação (o que ainda pode virar receita)
hoje = pd.Timestamp.today().normalize()
# Aberto = "Em negociação" (follow-up, proposta...) + "Reunião feita" (conversou e ainda não tem desfecho)
pipe = r[r["status_padrao"].isin(["Em negociação", "Reunião feita"])].copy()
pipe["Dias em aberto"] = (hoje - pipe["data_referencia"]).dt.days
pipe["Valor ponderado (R$)"] = pipe["valor"].fillna(0) * PROB_FECHAMENTO_NEGOCIACAO
pipe["Temperatura"] = np.select(
    [pipe["sinais_obs"].map(lambda l: "Provável perda" in l), pipe["status_padrao"].eq("Em negociação") | pipe["sinais_obs"].map(lambda l: "Promessa de fechamento" in l)],
    ["❄️ Fria (sinal de perda)", "🔥 Quente"], "🌤️ Morna")
pipe["Alerta"] = np.where(pipe["Dias em aberto"] > DIAS_ALERTA_PIPELINE, f"⚠️ sem atualização há +{DIAS_ALERTA_PIPELINE} dias",
                          np.where(pipe["Dias em aberto"].isna(), "sem data", "ok"))
pipe["_ord"] = pipe["Temperatura"].map({"🔥 Quente": 0, "🌤️ Morna": 1, "❄️ Fria (sinal de perda)": 2})
cols_pipe = ["cliente", "marcacao", "responsavel", "semana", "data_referencia", "status_original", "Temperatura", "sinais_texto",
             "observacoes", "Dias em aberto", "Alerta"] + (["valor", "Valor ponderado (R$)"] if TEM_VALOR else [])
pipeline = (pipe.sort_values(["_ord", "Dias em aberto"], ascending=[True, False])[cols_pipe]
                .rename(columns={"cliente": "Cliente", "marcacao": "Marcação", "responsavel": "Responsável", "semana": "Semana",
                                 "data_referencia": "Data", "status_original": "Etapa na planilha", "sinais_texto": "Sinais na observação",
                                 "observacoes": "Observação", "valor": "Valor (R$)"}))

# ---- Leitura das observações: quais objeções e sinais aparecem mais
exp = r[["cliente", "status_padrao", "semana", "observacoes", "sinais_obs"]].explode("sinais_obs").reset_index(drop=True)
exp = exp[exp["sinais_obs"].notna()]
tot_obs = r["observacoes"].notna().sum()
sinais_resumo = (exp.groupby("sinais_obs").agg(**{"Clientes": ("cliente", "nunique")}).reset_index()
                    .rename(columns={"sinais_obs": "Sinal na observação"}).sort_values("Clientes", ascending=False))
sinais_por_status = pd.crosstab(exp["sinais_obs"], exp["status_padrao"]).reset_index().rename(columns={"sinais_obs": "Sinal na observação"})
sinais_lista = (r[r["observacoes"].notna()][["cliente", "marcacao", "semana", "status_padrao", "sinais_texto", "observacoes"]]
                  .rename(columns={"cliente": "Cliente", "marcacao": "Marcação", "semana": "Semana", "status_padrao": "Status padronizado",
                                   "sinais_texto": "Sinais identificados", "observacoes": "Observação"}))
sinais_lista["Sinais identificados"] = sinais_lista["Sinais identificados"].fillna("—")

# ---- Listas de acompanhamento
noshow = r[r["status_padrao"].isin(["No-show", "Remarcado", "Cancelado"])][["cliente", "responsavel", "origem", "data_reuniao", "status_padrao", "status_original"]] \
            .rename(columns={"cliente": "Cliente", "responsavel": "Responsável", "origem": "Origem", "data_reuniao": "Data da reunião",
                             "status_padrao": "Status padronizado", "status_original": "Status na planilha"})

print(f"Perdas registradas: {len(perd)} | Em aberto: {len(pipe)} "
      f"({(pipe['Temperatura'] == '🔥 Quente').sum()} quentes, {(pipe['Temperatura'] == '🌤️ Morna').sum()} mornas, "
      f"{(pipe['Temperatura'] == '❄️ Fria (sinal de perda)').sum()} frias) | No-show/remarcadas/canceladas: {len(noshow)}")
display(sinais_resumo)
display(pipeline)

## 11. Cruzamento Meta × Comercial (funil completo e resultado por campanha)

In [ ]:
funil = pd.DataFrame({
    "Etapa": ["Impressões", "Cliques no link", "Leads / conversas (Meta)", "Reuniões agendadas (tráfego pago)",
              "Reuniões realizadas (tráfego pago)", "Clientes fechados (tráfego pago)"],
    "Quantidade": [IMP, CLQ, LEADS, pago["Reuniões agendadas"], pago["Reuniões realizadas"], pago["Fechados"]],
})
funil["Conversão da etapa anterior (%)"] = div(funil["Quantidade"], funil["Quantidade"].shift(1).fillna(0))
funil["Conversão desde impressões (%)"] = div(funil["Quantidade"], IMP)
funil["Custo por etapa (R$)"] = div(INV, funil["Quantidade"])

# Resultado por campanha: só é possível se a planilha de reuniões tiver a coluna "Campanha"
cruz = pd.DataFrame()
if rp["campanha"].notna().any():
    base_m = meta_camp.assign(chave=meta_camp["Campanha"].map(normalizar))[["chave", "Campanha", "Investimento (R$)", "Leads / resultados"]] \
                      .rename(columns={"Leads / resultados": "Leads Meta"})
    base_r = rp.assign(chave=rp["campanha"].fillna("(campanha não informada)").map(normalizar), campanha=rp["campanha"].fillna("(campanha não informada)"))
    com = base_r.groupby("chave").apply(contagens, include_groups=False).reset_index()
    nomes = base_r.groupby("chave")["campanha"].first()
    cruz = base_m.merge(com, on="chave", how="outer")
    cruz["Campanha"] = cruz["Campanha"].fillna(cruz["chave"].map(nomes) + "  (não achada na Meta)")
    cruz = cruz.drop(columns="chave").fillna(0)
    cols = ["Campanha", "Investimento (R$)", "Leads Meta", "Reuniões agendadas", "Reuniões realizadas", "No-show",
            "Fechados", "Perdidos", "Em negociação", "Receita fechada (R$)", "Valor em negociação (R$)"]
    cruz = cruz[cols].sort_values("Investimento (R$)", ascending=False).reset_index(drop=True)
    display(aplicar_taxas(cruz, TAXAS_CRUZADAS))
else:
    print("A planilha de reuniões não tem coluna 'Campanha' — o cruzamento por campanha foi pulado.\n"
          "Dica: registre de qual campanha veio cada lead para descobrir qual anúncio realmente vende.")
display(funil)

## 12. Resumo executivo e diagnóstico automático

In [ ]:
receita_pago = g(pago, "Receita fechada (R$)")
SEM_NOSHOW = (geral["No-show"] + geral["Remarcadas"] + geral["Canceladas"]) == 0
K = {  # chave: (rótulo, valor ou fórmula, formato, explicação, meta)
    "inv":     ("Investimento em tráfego pago", INV, "brl", "Soma de 'Valor usado' da Meta no período", None),
    "imp":     ("Impressões", IMP, "int", "Quantas vezes os anúncios apareceram", None),
    "alc":     ("Alcance (soma)", ALC, "int", "Pessoas alcançadas — somado entre anúncios, pode ter sobreposição", None),
    "clq":     ("Cliques no link", CLQ, "int", "Cliques que levaram ao destino do anúncio", None),
    "leads":   ("Leads / resultados (Meta)", LEADS, "int", "Coluna 'Resultados' da Meta (" + ", ".join(tipos_resultado) + ")", None),
    "ctr":     ("CTR no link", "={clq}/{imp}", "pct", "Cliques ÷ impressões", ("min", METAS["ctr"])),
    "cpc":     ("CPC", "={inv}/{clq}", "brl", "Investimento ÷ cliques", None),
    "cpm":     ("CPM", "={inv}/{imp}*1000", "brl", "Custo por mil impressões", None),
    "cpl":     ("Custo por lead/conversa (CPL)", "={inv}/{leads}", "brl", "Investimento ÷ leads", ("max", METAS["cpl_max"])),
    "ag":      ("Reuniões agendadas (total)", geral["Reuniões agendadas"], "int", "Todas as linhas do mês na planilha de reuniões", None),
    "real":    ("Reuniões realizadas (total)", geral["Reuniões realizadas"], "int", "Fechado + Perdido + Em negociação + Reunião feita", None),
    "ns":      ("No-show (total)", geral["No-show"], "int", "Cliente não compareceu", None),
    "rem":     ("Remarcadas + canceladas", geral["Remarcadas"] + geral["Canceladas"], "int", "Não aconteceram no dia marcado", None),
    "fut":     ("Ainda vão acontecer", geral["Ainda vão acontecer"], "int", "Status 'Agendado' — ficam fora das taxas", None),
    "fec":     ("Fechados (total)", geral["Fechados"], "int", "Status padronizado = Fechado", None),
    "per":     ("Perdidos (total)", geral["Perdidos"], "int", "Status padronizado = Perdido", None),
    "neg":     ("Em negociação / follow-up (total)", geral["Em negociação"], "int", "Follow-up, proposta, aguardando retorno", None),
    "sd":      ("Reunião feita, sem desfecho registrado", geral["Reunião feita sem desfecho"], "int", "Etapa 'Reunião Feita' — a conversa aconteceu, mas não diz se fechou ou perdeu", None),
    "agr":     ("Agendamento → realização", "={real}/({real}+{ns}+{rem})", "pct", "Realizadas ÷ (realizadas + no-show + remarcadas/canceladas)", None),
    "comp":    ("Comparecimento", "={real}/({real}+{ns})", "pct", "Realizadas ÷ (realizadas + no-show)" +
                ("" if SEM_NOSHOW is False else " — a planilha não registra no-show, por isso tende a 100%"),
                ("min", METAS["comparecimento"]) if SEM_NOSHOW is False else None),
    "assert":  ("⭐ ASSERTIVIDADE (fechados ÷ realizadas)", "={fec}/{real}", "pct", "De cada 100 reuniões feitas, quantas fecharam", ("min", METAS["assertividade"])),
    "win":     ("Win rate (fechados ÷ decididas)", "={fec}/({fec}+{per})", "pct",
                "Considera só quem já decidiu (fechou ou perdeu)" + (" — SEM PERDAS REGISTRADAS, por isso não é confiável" if geral["Perdidos"] == 0 else ""),
                ("min", METAS["win_rate"]) if geral["Perdidos"] > 0 else None),
    "fag":     ("Fechamento sobre agendadas", "={fec}/{ag}", "pct", "Fechados ÷ tudo que foi agendado", None),
    "rec":     ("Receita fechada (total)", g(geral, "Receita fechada (R$)"), "brl", "Soma do valor dos fechados", None),
    "tm":      ("Ticket médio", "={rec}/{fec}", "brl", "Receita ÷ fechados", None),
    "vneg":    ("Valor em negociação", g(geral, "Valor em negociação (R$)"), "brl", "Soma do valor das negociações abertas", None),
    "fcst":    ("Forecast ponderado do pipeline", f"={{vneg}}*{PROB_FECHAMENTO_NEGOCIACAO}", "brl", f"Valor em negociação × {PROB_FECHAMENTO_NEGOCIACAO:.0%} de probabilidade", None),
    "agp":     ("Reuniões agendadas (tráfego pago)", pago["Reuniões agendadas"], "int", "Reuniões cuja origem é tráfego pago", None),
    "realp":   ("Reuniões realizadas (tráfego pago)", pago["Reuniões realizadas"], "int", "Idem, só as realizadas", None),
    "fecp":    ("Fechados (tráfego pago)", pago["Fechados"], "int", "Clientes vindos do tráfego pago", None),
    "recp":    ("Receita fechada (tráfego pago)", receita_pago, "brl", "Receita dos clientes vindos do tráfego pago", None),
    "l2r":     ("Lead → reunião agendada", "={agp}/{leads}", "pct", "Reuniões agendadas pagas ÷ leads da Meta", ("min", METAS["lead_para_reuniao"])),
    "l2c":     ("Lead → cliente", "={fecp}/{leads}", "pct", "Fechados pagos ÷ leads da Meta", None),
    "cpr":     ("Custo por reunião realizada", "={inv}/{realp}", "brl", "Investimento ÷ reuniões realizadas pagas", None),
    "cac":     ("CAC (custo por cliente)", "={inv}/{fecp}", "brl", "Investimento ÷ clientes fechados do tráfego pago", ("max", METAS["cac_max"])),
    "roas":    ("ROAS", "={recp}/{inv}", "x", "Receita do tráfego pago ÷ investimento", ("min", METAS["roas"])),
    "roi":     ("ROI", "=({recp}-{inv})/{inv}", "pct", "(Receita − investimento) ÷ investimento", None),
}
BLOCOS = [("TRÁFEGO PAGO (META)", ["inv", "imp", "alc", "clq", "leads", "ctr", "cpc", "cpm", "cpl"]),
          ("COMERCIAL — TODAS AS ORIGENS", ["ag", "real", "ns", "rem", "fut", "fec", "neg", "sd", "per", "agr", "comp", "assert", "win", "fag", "rec", "tm", "vneg", "fcst"]),
          ("RESULTADO DO TRÁFEGO PAGO (META × REUNIÕES)", ["agp", "realp", "fecp", "recp", "l2r", "l2c", "cpr", "cac", "roas", "roi"])]
if not TEM_VALOR:   # sem coluna de valor: tira receita/ROAS em vez de mostrar zero
    SEM_VALOR_KPIS = {"rec", "tm", "vneg", "fcst", "recp", "roas", "roi"}
    BLOCOS = [(t, [k for k in ks if k not in SEM_VALOR_KPIS]) for t, ks in BLOCOS]
    K = {k: v for k, v in K.items() if k not in SEM_VALOR_KPIS}

# Avalia fórmulas em Python (para o notebook e para o diagnóstico)
VAL = {}
def avaliar(k):
    if k in VAL:
        return VAL[k]
    v = K[k][1]
    if isinstance(v, str):
        expr = re.sub(r"\{(\w+)\}", lambda mm: f"({avaliar(mm.group(1))})", v[1:])
        try:
            v = float(eval(expr))
            v = 0.0 if not np.isfinite(v) else v
        except ZeroDivisionError:
            v = 0.0
    VAL[k] = float(v)
    return VAL[k]
for k in K:
    avaliar(k)

def br(txt):
    # troca 1,234.56 -> 1.234,56 apenas dentro dos números
    # só mexe em números de dinheiro (R$ ...), porcentagem (...%) ou múltiplo (...x); nomes de campanha ficam intactos
    troca = lambda t: t.translate(str.maketrans(",.", ".,"))
    txt = re.sub(r"R\$ -?\d[\d,.]*", lambda mm: troca(mm.group(0)), txt)
    return re.sub(r"-?\d[\d,.]*(?=%|x\b)", lambda mm: troca(mm.group(0)), txt)

def fmt(v, f):
    return ({"brl": f"R$ {v:,.2f}", "pct": f"{v:.1%}", "int": f"{v:,.0f}", "x": f"{v:.2f}x"}[f])

# ---- Diagnóstico em texto
diag = []
def add(area, nivel, texto):
    diag.append({"Área": area, "Nível": nivel, "Diagnóstico": br(texto)})

for k, (rot, _, f, _, meta_k) in K.items():
    if meta_k:
        tipo, alvo = meta_k
        v = VAL[k]
        ok = v >= alvo if tipo == "min" else (v <= alvo and v > 0)
        add("Metas", "✅ Dentro da meta" if ok else "⚠️ Abaixo da meta",
            f"{rot}: {fmt(v, f)} (meta {'mínima' if tipo == 'min' else 'máxima'} {fmt(alvo, f)}).")

if not meta_camp.empty:
    mc = aplicar_taxas(meta_camp, TAXAS_META, INV)
    mc_l = mc[mc["Leads / resultados"] > 0]
    if len(mc_l):
        b, w = mc_l.nsmallest(1, "Custo por lead (R$)").iloc[0], mc_l.nlargest(1, "Custo por lead (R$)").iloc[0]
        add("Meta", "ℹ️", f"Menor CPL: '{b['Campanha']}' (R$ {b['Custo por lead (R$)']:,.2f}). Maior CPL: '{w['Campanha']}' (R$ {w['Custo por lead (R$)']:,.2f}).")
    zero = mc[(mc["Leads / resultados"] == 0) & (mc["Investimento (R$)"] > 0)]
    for _, z in zero.iterrows():
        add("Meta", "🔴 Atenção", f"'{z['Campanha']}' gastou R$ {z['Investimento (R$)']:,.2f} sem gerar nenhum resultado.")
if len(tipos_resultado) > 1:
    add("Meta", "⚠️ Atenção", f"Há tipos de resultado diferentes somados como lead ({', '.join(tipos_resultado)}). Compare CPL entre campanhas do mesmo tipo.")
if aviso_meta:
    add("Dados", "⚠️ Atenção", aviso_meta)
for a in AVISOS:
    add("Dados", "⚠️ Atenção", a)

if not cruz.empty:
    cz = aplicar_taxas(cruz, TAXAS_CRUZADAS)
    vend = cz[cz["Fechados"] > 0]
    if len(vend):
        b = vend.nsmallest(1, "CAC (R$)").iloc[0]
        add("Campanhas", "✅", f"Campanha com menor CAC: '{b['Campanha']}' — {b['Fechados']:.0f} cliente(s), CAC R$ {b['CAC (R$)']:,.2f}, ROAS {b['ROAS (x)']:.2f}x.")
    for _, z in cz[(cz["Fechados"] == 0) & (cz["Investimento (R$)"] > 0)].iterrows():
        add("Campanhas", "🔴 Atenção", f"'{z['Campanha']}' investiu R$ {z['Investimento (R$)']:,.2f}, gerou {z['Reuniões agendadas']:.0f} reunião(ões) e nenhum fechamento.")

pr = aplicar_taxas(por_resp, TAXAS_COMERCIAIS, g(geral, "Receita fechada (R$)"))
pr = pr[(pr["Reuniões realizadas"] > 0) & (pr["Responsável"] != "(sem responsável)")]
if len(pr) > 1:
    b, w = pr.iloc[pr["Assertividade - fechamento s/ realizadas (%)"].argmax()], pr.iloc[pr["Assertividade - fechamento s/ realizadas (%)"].argmin()]
    add("Equipe", "ℹ️", f"Maior assertividade: {b['Responsável']} ({b['Assertividade - fechamento s/ realizadas (%)']:.1%} em {b['Reuniões realizadas']:.0f} reuniões). "
                        f"Menor: {w['Responsável']} ({w['Assertividade - fechamento s/ realizadas (%)']:.1%} em {w['Reuniões realizadas']:.0f}).")
if len(motivos) and motivos.iloc[0]["Motivo"] != "(motivo não informado)":
    add("Perdas", "ℹ️", f"Principal motivo de perda: '{motivos.iloc[0]['Motivo']}' ({motivos.iloc[0]['% das perdas (%)']:.0%} das perdas, R$ {motivos.iloc[0]['Valor potencial perdido (R$)']:,.2f}).")
if (motivos["Motivo"] == "(motivo não informado)").any():
    q = motivos.loc[motivos["Motivo"] == "(motivo não informado)", "Quantidade"].iloc[0]
    add("Dados", "⚠️ Atenção", f"{q} perda(s) sem motivo registrado — preencher ajuda a entender o que trava a venda.")
paradas = (pipe["Dias em aberto"] > DIAS_ALERTA_PIPELINE).sum()
if paradas:
    extra = f" — somam R$ {pipe.loc[pipe['Dias em aberto'] > DIAS_ALERTA_PIPELINE, 'valor'].sum():,.2f}" if TEM_VALOR else ""
    add("Pipeline", "⚠️ Atenção", f"{paradas} oportunidade(s) em aberto há mais de {DIAS_ALERTA_PIPELINE} dias{extra}. Vale fazer follow-up ou dar como perdida.")
quentes = pipe[pipe["Temperatura"] == "🔥 Quente"]
if len(quentes):
    add("Pipeline", "🔥 Oportunidade", f"{len(quentes)} cliente(s) quentes para fechar agora: {', '.join(quentes['cliente'].astype(str))}.")
if geral["Perdidos"] == 0 and geral["Reuniões realizadas"] > 0:
    frias = (pipe["Temperatura"] == "❄️ Fria (sinal de perda)").sum()
    add("Perdas", "⚠️ Atenção", f"Nenhuma reunião está marcada como perdida. {geral['Reunião feita sem desfecho']:.0f} reuniões estão só como 'Reunião feita' "
        f"e {frias} delas têm sinal claro de perda na observação. Sem registrar perdas, a assertividade real fica escondida — "
        f"crie a etapa 'Perdido' na planilha (ou ligue CLASSIFICAR_PERDA_PELA_OBSERVACAO).")
if len(sinais_resumo):
    top = sinais_resumo[~sinais_resumo["Sinal na observação"].isin(["Pediu retorno / follow-up"])].head(3)
    if len(top):
        add("Objeções", "ℹ️", "Sinais mais frequentes nas observações: " +
            "; ".join(f"{row['Sinal na observação']} ({row['Clientes']} clientes)" for _, row in top.iterrows()) + ".")
fech_marc = r[(r["status_padrao"] == "Fechado") & (r["marcacao"] != "(sem marcação)")]
if len(fech_marc) and not MARCACOES_NAO_PAGAS:
    fecp_sem = pago["Fechados"] - len(fech_marc[fech_marc["trafego_pago"]])
    marcs = ", ".join(sorted(fech_marc["marcacao"].unique()))
    add("CAC", "⚠️ Confirmar", f"{len(fech_marc)} dos {geral['Fechados']:.0f} fechamentos têm a marcação '{marcs}' no nome. Hoje eles contam como tráfego pago "
        f"(CAC R$ {VAL.get('cac', 0):,.2f}). Se vieram de indicação/parceiro, o CAC real do anúncio é R$ {div(INV, fecp_sem):,.2f} "
        f"({fecp_sem:.0f} clientes). Para corrigir, preencha MARCACOES_NAO_PAGAS na configuração.")
if SEM_NOSHOW:
    add("Dados", "ℹ️", "Nenhuma reunião registrada como no-show, remarcada ou cancelada. Se isso aconteceu no mês, crie essas etapas "
        "na planilha — senão comparecimento e 'agendamento → realização' aparecem em 100% sem refletir a realidade.")
if not TEM_VALOR:
    add("Dados", "ℹ️", "A planilha não tem coluna de valor do contrato — receita, ticket médio, ROAS e ROI não foram calculados. "
        "Adicione uma coluna 'Valor' (mensalidade ou valor do contrato) para liberar esses indicadores.")
if r["responsavel"].eq("(sem responsável)").mean() > 0.5:
    add("Dados", "ℹ️", f"{r['responsavel'].eq('(sem responsável)').sum()} de {len(r)} reuniões estão sem 'Quem fez' — a comparação por vendedor fica incompleta.")
if geral["Ainda vão acontecer"]:
    add("Dados", "ℹ️", f"{geral['Ainda vão acontecer']:.0f} reunião(ões) ainda com status 'Agendado' — ficam fora das taxas até serem atualizadas.")
if geral["Não classificadas"]:
    add("Dados", "⚠️ Atenção", f"{geral['Não classificadas']:.0f} reunião(ões) com status não reconhecido. Veja a aba 'Qualidade dos Dados' e ajuste REGRAS_STATUS ou STATUS_MANUAL.")
sem_valor = r[(r["status_padrao"] == "Fechado") & (r["valor"].isna() | (r["valor"] == 0))] if TEM_VALOR else r.iloc[0:0]
if len(sem_valor):
    add("Dados", "⚠️ Atenção", f"{len(sem_valor)} fechamento(s) sem valor — receita, ticket médio e ROAS ficam subestimados.")
diagnostico = pd.DataFrame(diag)

for titulo, chaves in BLOCOS:
    print(f"\n=== {titulo} ===")
    for k in chaves:
        print(f"  {K[k][0]:<45} {br(fmt(VAL[k], K[k][2])):>18}")
display(diagnostico)

## 13. Qualidade dos dados
Tudo o que pode distorcer os números aparece aqui — para você corrigir na origem.

In [ ]:
problemas = []
def prob(tipo, df_, detalhe):
    for _, row in df_.iterrows():
        problemas.append({"Problema": tipo, "Cliente": row.get("cliente"), "Semana": row.get("semana_planilha"),
                          "Data como está na planilha": row.get("data_reuniao_texto"),
                          "Status na planilha": row.get("status_original"), "Observação": row.get("observacoes"), "Detalhe": detalhe})

prob("Status não reconhecido", r[r["status_padrao"] == "Não classificado"], "Adicione a palavra em REGRAS_STATUS ou STATUS_MANUAL")
prob("Sem status", r[r["status_padrao"] == "Sem status"], "Preencher o status na planilha")
prob("Fechado sem valor", sem_valor, "Receita subestimada")
if filtrar:
    prob("Sem data (fora da análise)", reun[reun["data_referencia"].isna()], "Linha ignorada por não ter data")
else:
    prob("Reunião sem data", r[r["data_reuniao"].isna()], "Entra nas contagens, mas não em 'dia da semana' nem em tempo de agendamento")
    prob("Data fora do mês", r[r["data_referencia"].notna() & (r["data_referencia"].dt.to_period("M") != periodo)],
         f"A data não é de {NOME_MES} — provável erro de digitação")
prob("Agendamento depois da reunião", r[r["data_agendamento"] > r["data_reuniao"]], "Data do agendamento é posterior à da reunião")
suspeito = r[(r["status_padrao"] == "Fechado") & r["sinais_obs"].map(
    lambda l: bool(set(l) & {"Promessa de fechamento", "Pediu retorno / follow-up", "Decide com cônjuge/sócio"}))]
prob("Fechado, mas a observação sugere que ainda não fechou", suspeito, "Confirme se o contrato foi mesmo assinado — pode estar inflando a assertividade")
dup = r[r["cliente"].notna() & r.duplicated(["cliente", "data_reuniao"], keep=False)]
prob("Possível duplicidade", dup, "Mesmo cliente e mesma data")
if "data_fechamento" in r:
    prob("Fechado sem data de fechamento", r[(r["status_padrao"] == "Fechado") & r["data_fechamento"].isna()], "Ciclo de venda não calculado para esta linha")
    prob("Data de fechamento antes da reunião", r[r["data_fechamento"] < r["data_reuniao"]], "Verificar datas")
qualidade = pd.DataFrame(problemas, columns=["Problema", "Cliente", "Semana", "Data como está na planilha", "Status na planilha", "Observação", "Detalhe"])
if len(suspeito):
    add("Dados", "⚠️ Confirmar", f"{len(suspeito)} cliente(s) marcados como Fechado têm observação de quem ainda vai decidir "
        f"({', '.join(suspeito['cliente'].astype(str))}). Se não fecharam, a assertividade real é {div(geral['Fechados'] - len(suspeito), geral['Reuniões realizadas']):.1%} "
        f"e não {VAL['assert']:.1%}.")
    diagnostico = pd.DataFrame(diag)
resumo_q = qualidade["Problema"].value_counts().rename_axis("Problema").reset_index(name="Ocorrências") if len(qualidade) else \
           pd.DataFrame({"Problema": ["Nenhum problema encontrado ✅"], "Ocorrências": [0]})
mapeamento = pd.concat([log_meta, log_reun], ignore_index=True)
display(resumo_q)

## 14. Geração da planilha Excel (formatada, com fórmulas vivas)

In [ ]:
# ---------- estilo ----------
AZUL, AZUL_CLARO, CINZA = "1F3864", "D9E2F3", "F2F2F2"
F_TIT = Font(name="Arial", bold=True, size=14, color=AZUL)
F_SUB = Font(name="Arial", italic=True, size=9, color="595959")
F_CAB = Font(name="Arial", bold=True, color="FFFFFF", size=10)
F_TXT = Font(name="Arial", size=10)
F_TOT = Font(name="Arial", bold=True, size=10)
FILL_CAB, FILL_TOT, FILL_BLOCO = PatternFill("solid", fgColor=AZUL), PatternFill("solid", fgColor=AZUL_CLARO), PatternFill("solid", fgColor=CINZA)
FILL_OK, FILL_RUIM = PatternFill("solid", fgColor="E2EFDA"), PatternFill("solid", fgColor="FCE4D6")
BORDA = Border(*(Side(style="thin", color="BFBFBF"),) * 4)
FMT = {"brl": '"R$" #,##0.00;-"R$" #,##0.00;"-"', "pct": '0.0%;-0.0%;"-"', "int": '#,##0;-#,##0;"-"',
       "x": '0.00"x";-0.00"x";"-"', "dec": '#,##0.0;-#,##0.0;"-"', "data": "dd/mm/yyyy"}

def formato_coluna(nome, serie=None):
    n = nome.lower()
    if "(r$)" in n: return FMT["brl"]
    if "(%)" in n: return FMT["pct"]
    if "(x)" in n: return FMT["x"]
    if "(dias)" in n or "dias" in n: return FMT["dec"]
    if serie is not None and pd.api.types.is_datetime64_any_dtype(serie): return FMT["data"]
    if serie is not None and pd.api.types.is_numeric_dtype(serie): return FMT["int"] if (serie.dropna() % 1 == 0).all() else FMT["dec"]
    return None

def valor_excel(v):
    if v is None or (isinstance(v, float) and np.isnan(v)) or v is pd.NA or v is pd.NaT: return None
    if isinstance(v, np.generic): return v.item()
    if isinstance(v, pd.Timestamp): return v.to_pydatetime()
    return v

def cabecalho(ws, titulo, subtitulo=None):
    ws["A1"] = titulo; ws["A1"].font = F_TIT
    ws["A2"] = subtitulo or f"Período: {NOME_MES}  •  Gerado em {datetime.now():%d/%m/%Y %H:%M}"; ws["A2"].font = F_SUB
    ws.sheet_view.showGridLines = False
    return 4

def escrever_tabela(ws, df, linha, titulo=None, taxas=None, total=True, nao_somar=(), col=1):
    # Escreve df a partir de 'linha'. Colunas de taxa viram FÓRMULAS do Excel. Retorna próxima linha livre.
    if titulo:
        ws.cell(linha, col, titulo).font = Font(name="Arial", bold=True, size=11, color=AZUL); linha += 1
    if df is None or df.empty:
        ws.cell(linha, col, "Sem dados para este bloco.").font = F_SUB
        return linha + 2
    taxas = [t for t in (taxas or []) if all(c in df for c in t["num"]) and (t["den"] == "__TOTAL__" or all(c in df for c in t["den"]))]
    df = df.drop(columns=[t["nome"] for t in taxas if t["nome"] in df])
    cols = list(df.columns) + [t["nome"] for t in taxas]
    L = {c: get_column_letter(col + i) for i, c in enumerate(cols)}
    for i, c in enumerate(cols):
        cel = ws.cell(linha, col + i, c); cel.font, cel.fill, cel.border = F_CAB, FILL_CAB, BORDA
        cel.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[linha].height = 32
    cab, ini, fim = linha, linha + 1, linha + len(df)
    lin_tot = fim + 1

    def formula(t, rr):
        num = "+".join(f"{L[c]}{rr}" for c in t["num"])
        den = f"{L[t['num'][0]]}${lin_tot}" if t["den"] == "__TOTAL__" else "+".join(f"{L[c]}{rr}" for c in t["den"])
        mult = f"*{t['mult']}" if t.get("mult") else ""
        return f"=IFERROR(({num})/({den}){mult},0)"

    for j, (_, row) in enumerate(df.iterrows()):
        rr = ini + j
        for i, c in enumerate(df.columns):
            cel = ws.cell(rr, col + i, valor_excel(row[c]))
            cel.font, cel.border = F_TXT, BORDA
            f = formato_coluna(c, df[c])
            if f: cel.number_format = f
            if j % 2: cel.fill = FILL_BLOCO
        for t in taxas:
            cel = ws.cell(rr, col + cols.index(t["nome"]), formula(t, rr))
            cel.font, cel.border, cel.number_format = F_TXT, BORDA, formato_coluna(t["nome"]) or FMT["pct"]
            if j % 2: cel.fill = FILL_BLOCO
    if total:
        for i, c in enumerate(cols):
            cel = ws.cell(lin_tot, col + i); cel.font, cel.fill, cel.border = F_TOT, FILL_TOT, BORDA
            if i == 0:
                cel.value = "TOTAL"
            elif c in df and pd.api.types.is_numeric_dtype(df[c]) and c not in nao_somar:
                cel.value = f"=SUM({L[c]}{ini}:{L[c]}{fim})"; cel.number_format = formato_coluna(c, df[c]) or FMT["int"]
            elif c not in df:
                t = next(t for t in taxas if t["nome"] == c)
                cel.value = formula(t, lin_tot); cel.number_format = formato_coluna(c) or FMT["pct"]
    ultima = lin_tot if total else fim
    ws.auto_filter.ref = f"{get_column_letter(col)}{cab}:{get_column_letter(col + len(cols) - 1)}{fim}" if not ws.auto_filter.ref else ws.auto_filter.ref
    return ultima + 2

def ajustar_larguras(ws, minimo=10, maximo=55):
    larg = {}
    for row in ws.iter_rows(min_row=3):
        for c in row:
            if c.value is not None and not str(c.value).startswith("="):
                larg[c.column_letter] = max(larg.get(c.column_letter, 0), len(str(c.value)))
    for letra, w in larg.items():
        ws.column_dimensions[letra].width = max(minimo, min(maximo, w * 0.95 + 2))

print("Funções de escrita prontas ✔")

In [ ]:
wb = Workbook()

# ============ 1. RESUMO EXECUTIVO ============
ws = wb.active; ws.title = "Resumo Executivo"
lin = cabecalho(ws, f"Assertividade Comercial — {NOME_MES}")
for i, h in enumerate(["Indicador", "Valor", "Como é calculado", "Meta", "Situação"], 1):
    c = ws.cell(lin, i, h); c.font, c.fill, c.border = F_CAB, FILL_CAB, BORDA
lin += 1
REF = {}
# primeiro passa: reserva linha de cada KPI
cursor = lin
for titulo, chaves in BLOCOS:
    cursor += 1
    for k in chaves:
        REF[k] = f"B{cursor}"; cursor += 1
    cursor += 1
for titulo, chaves in BLOCOS:
    c = ws.cell(lin, 1, titulo); c.font = Font(name="Arial", bold=True, color=AZUL, size=11)
    for i in range(1, 6): ws.cell(lin, i).fill = FILL_TOT
    lin += 1
    for k in chaves:
        rot, v, f, expl, meta_k = K[k]
        ws.cell(lin, 1, rot).font = Font(name="Arial", size=10, bold=(k == "assert"))
        if isinstance(v, str):
            v = "=IFERROR(" + re.sub(r"\{(\w+)\}", lambda mm: REF[mm.group(1)], v[1:]) + ",0)"
        cel = ws.cell(lin, 2, valor_excel(v)); cel.number_format = FMT[f]; cel.font = Font(name="Arial", size=10, bold=True)
        ws.cell(lin, 3, expl).font = F_SUB
        if meta_k:
            tipo, alvo = meta_k
            cm = ws.cell(lin, 4, alvo); cm.number_format = FMT[f]; cm.font = Font(name="Arial", size=10, color="0000FF")
            cond = f"B{lin}>=D{lin}" if tipo == "min" else f"AND(B{lin}<=D{lin},B{lin}>0)"
            ws.cell(lin, 5, f'=IF({cond},"✅ Dentro da meta","⚠️ Fora da meta")').font = F_TXT
            ok = VAL[k] >= alvo if tipo == "min" else (0 < VAL[k] <= alvo)
            for i in (1, 2, 5): ws.cell(lin, i).fill = FILL_OK if ok else FILL_RUIM
        for i in range(1, 6): ws.cell(lin, i).border = BORDA
        lin += 1
    lin += 1
ws.cell(lin, 1, "Metas em azul são editáveis — a coluna Situação recalcula sozinha. Metas vêm da célula de configuração do notebook.").font = F_SUB
for letra, w in zip("ABCDE", [44, 18, 62, 14, 20]): ws.column_dimensions[letra].width = w
ws.freeze_panes = "A5"

# ============ 2. DIAGNÓSTICO ============
ws = wb.create_sheet("Diagnóstico")
lin = cabecalho(ws, "Diagnóstico automático — o que os números estão dizendo")
lin = escrever_tabela(ws, diagnostico, lin, total=False)
ws.column_dimensions["A"].width, ws.column_dimensions["B"].width, ws.column_dimensions["C"].width = 14, 20, 130
for row in ws.iter_rows(min_row=5):
    for c in row: c.alignment = Alignment(wrap_text=True, vertical="top")

# ============ 3. FUNIL ============
ws = wb.create_sheet("Funil Completo")
lin = cabecalho(ws, "Funil completo: do anúncio ao cliente (somente tráfego pago)")
ws.cell(lin, 1, "Investimento (R$)").font = F_TOT
ws.cell(lin, 2, INV).number_format = FMT["brl"]; lin += 2
f_ini = lin + 1
for i, h in enumerate(["Etapa", "Quantidade", "Conversão da etapa anterior (%)", "Conversão desde impressões (%)", "Custo por etapa (R$)"], 1):
    c = ws.cell(lin, i, h); c.font, c.fill, c.border = F_CAB, FILL_CAB, BORDA; c.alignment = Alignment(wrap_text=True, horizontal="center")
for j, row in funil.iterrows():
    rr = f_ini + j
    ws.cell(rr, 1, row["Etapa"]); ws.cell(rr, 2, valor_excel(row["Quantidade"])).number_format = FMT["int"]
    ws.cell(rr, 3, "" if j == 0 else f"=IFERROR(B{rr}/B{rr-1},0)").number_format = FMT["pct"]
    ws.cell(rr, 4, f"=IFERROR(B{rr}/$B${f_ini},0)").number_format = '0.000%'
    ws.cell(rr, 5, f"=IFERROR($B$4/B{rr},0)").number_format = FMT["brl"]
    for i in range(1, 6): ws.cell(rr, i).border, ws.cell(rr, i).font = BORDA, F_TXT
graf = BarChart(); graf.type = "bar"; graf.title = "Funil (etapas após o clique)"; graf.style = 10
graf.add_data(Reference(ws, min_col=2, min_row=f_ini + 1, max_row=f_ini + 5), titles_from_data=False)
graf.set_categories(Reference(ws, min_col=1, min_row=f_ini + 1, max_row=f_ini + 5))
graf.legend = None; graf.y_axis.majorGridlines = None; graf.x_axis.scaling.orientation = "maxMin"
graf.height, graf.width = 8, 18
ws.add_chart(graf, f"A{f_ini + 8}")
ws.cell(f_ini + 7, 1, "Obs.: impressões ficam fora do gráfico porque a escala esconderia as outras etapas.").font = F_SUB
for letra, w in zip("ABCDE", [40, 16, 20, 20, 18]): ws.column_dimensions[letra].width = w

# ============ 4. META ============
ws = wb.create_sheet("Meta - Campanhas")
lin = cabecalho(ws, "Meta Ads — desempenho por campanha")
lin = escrever_tabela(ws, meta_camp, lin, taxas=TAXAS_META, nao_somar=["Tipo de resultado"])
ws.cell(lin, 1, "Alcance e frequência no TOTAL são somas entre campanhas — podem conter pessoas repetidas.").font = F_SUB
ajustar_larguras(ws); ws.freeze_panes = "B5"

# só cria as abas de conjunto/anúncio se a exportação da Meta vier nesse nível
if (m["conjunto"] != "(sem informação)").any():
    ws = wb.create_sheet("Meta - Conjuntos")
    lin = cabecalho(ws, "Meta Ads — desempenho por conjunto de anúncios")
    escrever_tabela(ws, meta_conj, lin, taxas=TAXAS_META); ajustar_larguras(ws); ws.freeze_panes = "C5"
if (m["anuncio"] != "(sem informação)").any():
    ws = wb.create_sheet("Meta - Anúncios")
    lin = cabecalho(ws, "Meta Ads — desempenho por anúncio (criativo)")
    escrever_tabela(ws, meta_anun, lin, taxas=TAXAS_META); ajustar_larguras(ws); ws.freeze_panes = "D5"

# ============ 5. CAMPANHA x VENDAS ============
ws = wb.create_sheet("Campanha x Vendas")
lin = cabecalho(ws, "Qual campanha realmente vende — Meta cruzada com as reuniões")
if cruz.empty:
    ws.cell(lin, 1, "A planilha de reuniões não tem a coluna 'Campanha'. Registre de qual campanha veio cada lead para liberar esta aba.").font = F_SUB
else:
    escrever_tabela(ws, cruz, lin, taxas=TAXAS_CRUZADAS)
ajustar_larguras(ws); ws.freeze_panes = "B5"

# ============ 6. COMERCIAL ============
ws = wb.create_sheet("Status das Reuniões")
lin = cabecalho(ws, "Distribuição dos status das reuniões")
lin = escrever_tabela(ws, por_status.drop(columns="% do total (%)"), lin, "Status padronizado",
                      taxas=[{"nome": "% do total (%)", "num": ["Quantidade"], "den": "__TOTAL__"}])
lin = escrever_tabela(ws, mapa_status, lin, "Como cada texto da planilha foi classificado")
escrever_tabela(ws, matriz_resp, lin, "Status × responsável")
ajustar_larguras(ws)

for nome_aba, tab, titulo, rot in [
    ("Por Responsável", por_resp, "Assertividade por responsável (closer / vendedor)", "B5"),
    ("Por Origem", por_origem, "Assertividade por origem do lead", "B5"),
    ("Por Produto", por_produto, "Assertividade por produto / serviço", "B5"),
    ("Por Marcação", por_marcacao, "Assertividade por marcação no nome do cliente (ex.: '- Parceiro')", "B5"),
    ("Por Semana", por_semana, "Evolução semana a semana", "B5"),
    ("Por Dia da Semana", por_dia, "Assertividade por dia da semana da reunião", "B5"),
]:
    if tab is None or tab.empty or (nome_aba == "Por Origem" and not tem_origem):
        continue   # não cria aba vazia
    ws = wb.create_sheet(nome_aba)
    lin = cabecalho(ws, titulo)
    escrever_tabela(ws, tab, lin, taxas=TAXAS_COMERCIAIS)
    ajustar_larguras(ws, maximo=24); ws.freeze_panes = rot

ws = wb.create_sheet("Ciclo de Venda")
lin = cabecalho(ws, "Ciclo de venda — quanto tempo leva para fechar")
lin = escrever_tabela(ws, tempo_agenda, lin, "Tempo entre o agendamento e a reunião", total=False)
lin = escrever_tabela(ws, ciclo, lin, "Reunião → fechamento (precisa da coluna 'Data de fechamento')", total=False)
lin = escrever_tabela(ws, ciclo_resp, lin, "Reunião → fechamento por responsável", total=False)
if not fech.empty and "Dias até fechar" in fech:
    lista = fech[["cliente", "responsavel", "origem", "data_reuniao", "data_fechamento", "Dias até fechar", "valor"]].rename(
        columns={"cliente": "Cliente", "responsavel": "Responsável", "origem": "Origem", "data_reuniao": "Data da reunião",
                 "data_fechamento": "Data de fechamento", "valor": "Valor (R$)"}).sort_values("Dias até fechar")
    escrever_tabela(ws, lista, lin, "Todos os fechamentos do mês", nao_somar=["Dias até fechar"])
ajustar_larguras(ws)

ws = wb.create_sheet("Motivos de Perda")
lin = cabecalho(ws, "Por que os clientes não fecharam")
lin = escrever_tabela(ws, motivos.drop(columns="% das perdas (%)"), lin,
                      taxas=[{"nome": "% das perdas (%)", "num": ["Quantidade"], "den": "__TOTAL__"}])
if not perd.empty:
    lista = perd[["cliente", "responsavel", "origem", "data_reuniao", "motivo_perda", "valor"]].rename(
        columns={"cliente": "Cliente", "responsavel": "Responsável", "origem": "Origem", "data_reuniao": "Data da reunião",
                 "motivo_perda": "Motivo", "valor": "Valor (R$)"})
    escrever_tabela(ws, lista, lin, "Lista de perdas")
ajustar_larguras(ws)

ws = wb.create_sheet("Pipeline em Aberto")
lin = cabecalho(ws, "Oportunidades em aberto (Follow-up + Reunião feita) — ordenadas por temperatura")
escrever_tabela(ws, pipeline, lin, nao_somar=["Dias em aberto"], total=TEM_VALOR)
ajustar_larguras(ws, maximo=70); ws.freeze_panes = "B5"
for row in ws.iter_rows(min_row=5):
    for c in row: c.alignment = Alignment(wrap_text=True, vertical="top")

ws = wb.create_sheet("Observações e Objeções")
lin = cabecalho(ws, "O que as observações das reuniões revelam (objeções, sinais de compra e de perda)")
lin = escrever_tabela(ws, sinais_resumo, lin, "Sinais mais frequentes", total=False)
lin = escrever_tabela(ws, sinais_por_status, lin, "Sinal × status da reunião")
escrever_tabela(ws, sinais_lista, lin, "Todas as observações, cliente a cliente", total=False)
ajustar_larguras(ws, maximo=80)
for row in ws.iter_rows(min_row=5):
    for c in row: c.alignment = Alignment(wrap_text=True, vertical="top")

ws = wb.create_sheet("No-show e Remarcações")
lin = cabecalho(ws, "Reuniões que não aconteceram — lista para recuperar")
escrever_tabela(ws, noshow, lin, total=False); ajustar_larguras(ws)

# ============ 7. BASES TRATADAS ============
ws = wb.create_sheet("Base Reuniões (tratada)")
lin = cabecalho(ws, "Base de reuniões do mês, já padronizada (use para conferir qualquer número)")
base = r.drop(columns=["status", "status_ord", "sinais_obs"], errors="ignore").rename(columns={"valor": "valor (R$)"})
if not TEM_VALOR:
    base = base.drop(columns=["valor (R$)"], errors="ignore")
base = base.dropna(axis=1, how="all")
escrever_tabela(ws, base, lin, total=False); ajustar_larguras(ws, maximo=35); ws.freeze_panes = "A5"

ws = wb.create_sheet("Base Meta (tratada)")
lin = cabecalho(ws, "Base da Meta do período, já padronizada")
escrever_tabela(ws, m.rename(columns={"investimento": "investimento (R$)"}), lin, total=False); ajustar_larguras(ws, maximo=40)

# ============ 8. QUALIDADE + GLOSSÁRIO ============
ws = wb.create_sheet("Qualidade dos Dados")
lin = cabecalho(ws, "Qualidade dos dados — o que pode estar distorcendo os números")
lin = escrever_tabela(ws, resumo_q, lin, "Resumo", total=False)
lin = escrever_tabela(ws, qualidade, lin, "Detalhe linha a linha", total=False)
escrever_tabela(ws, mapeamento, lin, "Como as colunas dos seus arquivos foram reconhecidas", total=False)
ajustar_larguras(ws)

ws = wb.create_sheet("Glossário")
lin = cabecalho(ws, "Glossário — definição de cada indicador")
gl = pd.DataFrame([(v[0], v[3]) for v in K.values()] + [
    ("Reunião realizada", "Status Fechado, Perdido, Em negociação/Follow-up ou Reunião feita (a conversa aconteceu)"),
    ("Temperatura (pipeline)", "🔥 Quente = Follow-up ou promessa de fechamento na observação; ❄️ Fria = sinal de perda na observação; 🌤️ Morna = o resto"),
    ("Marcação", "Texto depois do hífen no nome do cliente (ex.: 'Wagner - Parceiro' → Parceiro)"),
    ("Status 'Agendado'", "Reunião futura; não entra em nenhuma taxa até ser atualizada"),
    ("Tráfego pago", "Origem contém: " + ", ".join(ORIGENS_TRAFEGO_PAGO) + (" (ou linha com campanha preenchida)" if tem_origem else " — sem coluna de origem, todas consideradas pagas")),
    ("Status padronizado", "Texto livre da planilha convertido pelas regras do notebook (aba Status das Reuniões mostra a conversão)"),
], columns=["Indicador", "Definição"])
escrever_tabela(ws, gl, lin, total=False)
ws.column_dimensions["A"].width, ws.column_dimensions["B"].width = 42, 110

wb.calculation.fullCalcOnLoad = True
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
ARQUIVO_SAIDA = PASTA_SAIDA / f"Assertividade_Comercial_{MES_REFERENCIA}.xlsx"
wb.save(ARQUIVO_SAIDA)
print(f"✅ Relatório salvo em: {ARQUIVO_SAIDA.resolve()}")
print("Abas:", ", ".join(wb.sheetnames))

## 15. Gráficos rápidos no notebook (opcional)

In [ ]:
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 3, figsize=(18, 5))
    f2 = funil.iloc[1:]
    ax[0].barh(f2["Etapa"], f2["Quantidade"], color="#1F3864"); ax[0].invert_yaxis(); ax[0].set_title("Funil (após o clique)")
    for i, v in enumerate(f2["Quantidade"]): ax[0].text(v, i, f" {v:,.0f}", va="center")
    ax[1].pie(por_status["Quantidade"], labels=por_status["Status padronizado"], autopct="%1.0f%%", startangle=90); ax[1].set_title("Status das reuniões")
    pr_ = aplicar_taxas(por_resp, TAXAS_COMERCIAIS, g(geral, "Receita fechada (R$)"))
    ax[2].bar(pr_["Responsável"], pr_["Assertividade - fechamento s/ realizadas (%)"] * 100, color="#2E75B6")
    ax[2].axhline(METAS["assertividade"] * 100, ls="--", color="red", label="meta"); ax[2].legend()
    ax[2].set_title("Assertividade por responsável (%)")
    plt.tight_layout(); plt.show()
except ImportError:
    print("matplotlib não instalado — pule esta célula ou rode: pip install matplotlib")